# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Михайло Білоусов'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 2       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 2   ·   Михайло Білоусов   ·   ШІ-32

Ваші вісім стовпців:
   1. Телефон
   2. Супутній
   3. ЛН
   4. Дата розтину
   5. КТ
   6. Фібрин1
   7. Сечов2
   8. ФЕРИТ1

Ваша пара для аналізу зв’язку: ЛН та помер/жив

Ваше додаткове питання:
   Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Супутній,ЛН,Дата розтину,КТ,Фібрин1,Сечов2,ФЕРИТ1
0,0958469788 дружина Ганна,"ІХС, КС, СН1, НВ ФК2",ІІІ-IV,2022-03-11,20-30%,4,6.5,NaN
1,"0671778486, 0994887663 дружина Оксана","ЦД, ІІ тип",ІІ-ІІІ,NaT,30-35%,5,4,839.0
2,"0958768045, чоловік Степан 0951784726","ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глю...",ІІ,NaT,70-80%,5.29,10.1,NaN
3,0959819331. дочка Валентина 0960845205,"ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусов...",ІІІ-IV,NaT,0.8,2.5,12.5,NaN
4,"0952294866, дочка Дарина 0678360161",ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3,ІІ-ІІІ,NaT,NaN,4.29,7.4,NaN
5,0959922983 сестра Софія,"ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ",ІІ-ІІІ,2021-12-26,30-40%,5,21.4,NaN
6,"0998073702, дружина Наталія 0998382226","ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений",ІІ-ІІІ,NaT,60-65%,3.24,6.1,NaN
7,"0637258771, син Сергій 0968413301","ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/...",І,NaT,до 17%,5.04,7.6,NaN


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()
print('-'*74)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Телефон       120 non-null    object        
 1   Супутній      115 non-null    str           
 2   ЛН            121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   КТ            44 non-null     object        
 5   Фібрин1       115 non-null    object        
 6   Сечов2        102 non-null    object        
 7   ФЕРИТ1        2 non-null      float64       
dtypes: datetime64[us](1), float64(1), object(4), str(2)
memory usage: 7.7+ KB
--------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 77 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   помер/жив       121 non-null    int64         
 1 

In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Супутній
    непорожніх 115 з 121 | унікальних 91 | типи: str×115
    приклади: ["'ІХС, КС, СН1, НВ ФК2'", "'ЦД, ІІ тип'", "'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози'", "'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв '"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'7

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'Телефон': ('ідентифікатор', 'виділяє конкретний випадок у датасеті'),
    'Супутній': ('номінальна', 'визначає конкретне захворювання, можна порівняти тільки наявність чи відсутність'),
    'ЛН': ('порядкова', 'легенева недостатність, можна порівняти ступені'),
    'Дата розтину': ('інтервальна', 'timestamp, також витік даних бо розтин можливий лише для померлих'),
    'КТ': ('відношень', 'відсоток ураження? за дослідженням кт'),
    'Фібрин1': ('абсолютна', 'вимір кількості речовини, є природня од вимірювання'),
    'Сечов2': ('абсолютна', 'вимір кількості речовини, є природня од вимірювання'),
    'ФЕРИТ1': ('абсолютна', 'вимір кількості речовини, є природня од вимірювання'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,виділяє конкретний випадок у датасеті
1,Супутній,номінальна,"визначає конкретне захворювання, можна порівняти тільки ..."
2,ЛН,порядкова,"легенева недостатність, можна порівняти ступені"
3,Дата розтину,інтервальна,"timestamp, також витік даних бо розтин можливий лише для..."
4,КТ,відношень,відсоток ураження? за дослідженням кт
5,Фібрин1,абсолютна,"вимір кількості речовини, є природня од вимірювання"
6,Сечов2,абсолютна,"вимір кількості речовини, є природня од вимірювання"
7,ФЕРИТ1,абсолютна,"вимір кількості речовини, є природня од вимірювання"


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації
    р['mode'] = з.mode().iloc[0] if len(з) else None
    р['freq'] = з.value_counts().to_dict()
    if pd.api.types.is_datetime64_any_dtype(з):
        vals = з.astype('int64').astype(float)
    else:
        num = pd.to_numeric(з, errors='coerce')
        vals = num.astype(float) if num.notna().all() else None
    if ш == 'порядкова':
        if vals is not None:
            р['median'] = vals.median()
            р['quantiles'] = vals.quantile([0.25, 0.5, 0.75]).to_dict()
    elif ш in ('інтервальна', 'відношень', 'абсолютна'):
        if vals is not None:
            р['mean'] = vals.mean()
            р['std'] = vals.std()
            if ш in ('відношень', 'абсолютна') and vals.mean():
                р['cv'] = vals.std() / vals.mean()
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Супутній  (номінальна)
    {'непорожніх': 115, 'унікальних': 91, 'mode': 'ІХС ДК ГХст2', 'freq': {'ІХС ДК ГХст2': 7, 'ІХС.ДК. ГХ ІІ': 5, 'ІХС ДК ГХст2.': 4, 'ІХС.ДК .Атеросклероз аорти. Первинно зморщені нирки': 3, 'Хрон гастродуоденіт': 2, 'ІХС ДК ГХст2 СНІІа': 2, 'ІХС ДК ГХст3': 2, 'ІХС.ДК. ХОЗЛ. ГХ ІІ': 2, 'ІХС.ДК. ГХ ІІ. ЦД ': 2, 'ІХС ДК ГХ ІІ СНІІа ФК3': 2, 'ІХС ДК ГХст2. ЦД': 2, 'ІХС.ДК. ЦД. Атеросклероз аорти': 2, 'ІХС ДК ГЛШ ст2 СН ІІА ': 2, 'ІХС, КС, СН1, НВ ФК2': 1, 'ЦД, ІІ тип': 1, 'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози': 1, 'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв ': 1, 'ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3': 1, 'ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ': 1, 'ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений': 1, 'ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/дуоденіт': 1, 'ІХС ДК ГХ2 ГЛШст2 КВР3 СН2 ФК3, ЦД2, Cr прав молоч залози після промен терап': 1, 'І

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
descr = df[МОЇ].describe(include='all').T
descr

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Супутній,115,91,ІХС ДК ГХст2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Фібрин1,115.0,87.0,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов2,102.0,66.0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ФЕРИТ1,2.0,NaN,NaN,NaN,555.05,271.1,413.075,555.05,697.025,839.0,401.565941


In [10]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Дата розтину', 'mean', descr['mean']['Дата розтину'], 'неможливо порахувати середній timestamp'),
    ('Телефон', 'freq', descr['freq']['Телефон'], 'усі ідентифікатори унікальні'),
    ('Супутній', 'top', descr['top']['Супутній'], 'ця колонка містить декілька значень в одному рядку, щоб це мало сенс треба розбивати'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Дата розтину,mean,2022-04-06 13:22:37.377049,неможливо порахувати середній timestamp
1,Телефон,freq,1,усі ідентифікатори унікальні
2,Супутній,top,ІХС ДК ГХст2,"ця колонка містить декілька значень в одному рядку, щоб ..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ЛН × помер/жив

ЛН: шкала — порядкова
помер/жив: шкала — не у вашому списку

       ЛН  помер/жив
0  ІІІ-IV          1
1  ІІ-ІІІ          2
2      ІІ          2
3  ІІІ-IV          2
4  ІІ-ІІІ          2
5  ІІ-ІІІ          1


In [12]:
ОБРАНА_МІРА = 'коефіцієнт Крамера (V)'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'порядкова х номінальна шкали, тому коефіцієнт крамера'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association

t = pd.crosstab(df[A], df[B])
chi2, p, dof, exp = chi2_contingency(t)
n = t.to_numpy().sum()
v = association(t, method='cramer')
print(t)
print('n =', int(n), '| chi2 =', round(chi2, 3), '| df =', dof, '| p =', round(p, 4))
print('Cramer V =', round(v, 3))
print('мінімальне очікуване =', round(exp.min(), 2))

помер/жив   1   2
ЛН               
І           0  12
І-ІІ        0   9
ІІ          0  21
ІІ-ІІІ     49  10
ІІІ        11   6
ІІІ-IV      1   2
n = 121 | chi2 = 69.58 | df = 5 | p = 0.0
Cramer V = 0.758
мінімальне очікуване = 1.49


In [13]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
from scipy.stats import pearsonr

codes = pd.factorize(df[A])[0]
y = pd.to_numeric(df[B], errors='coerce')
r, p = pearsonr(codes, y)
rc, pc = pearsonr(codes.max() - codes, y)
print('Pearson на довільних кодах ЛН            :', round(r, 3), '| p =', round(p, 4))

Pearson на довільних кодах ЛН            : 0.395 | p = 0.0


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'Телефон': 'ідентифікатор — у модель не беремо взагалі',
    'Супутній': 'номінальна — one-hot кодування, порядок категорій не задаємо',
    'ЛН': 'порядкова — цілочисельні коди із заданим порядком ступенів',
    'Дата розтину': 'інтервальна — переходимо до тривалості (днів від опорної дати)',
    'КТ': 'відношень — узгоджуємо формати й одиниці, далі z-стандартизація',
    'Фібрин1': 'абсолютна — z-стандартизація після перевірки варіативності',
    'Сечов2': 'абсолютна — z-стандартизація після перевірки варіативності',
    'ФЕРИТ1': 'абсолютна — z-стандартизація; n=2, тому ознака фактично непридатна',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,ідентифікатор — у модель не беремо взагалі
1,Супутній,номінальна,"номінальна — one-hot кодування, порядок категорій не зад..."
2,ЛН,порядкова,порядкова — цілочисельні коди із заданим порядком ступенів
3,Дата розтину,інтервальна,інтервальна — переходимо до тривалості (днів від опорної...
4,КТ,відношень,"відношень — узгоджуємо формати й одиниці, далі z-стандар..."
5,Фібрин1,абсолютна,абсолютна — z-стандартизація після перевірки варіативності
6,Сечов2,абсолютна,абсолютна — z-стандартизація після перевірки варіативності
7,ФЕРИТ1,абсолютна,"абсолютна — z-стандартизація; n=2, тому ознака фактично ..."


In [15]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
ln_order = {'І': 1, 'І-ІІ': 2, 'ІІ': 3, 'ІІ-ІІІ': 4, 'ІІІ': 5, 'ІІІ-IV': 6}
ln_codes = df['ЛН'].map(ln_order)
print('ЛН (порядкова) -> коди із заданим порядком:', ln_order)
print('розподіл кодів:', ln_codes.value_counts().sort_index().to_dict())

raw = df['КТ'].astype(str).str.strip()
kt_point = pd.to_numeric(raw.str.replace('%', '', regex=False), errors='coerce')
kt_range = raw.str.extract(r'(\d+(?:[.,]\d+)?)\s*[-–]\s*(\d+(?:[.,]\d+)?)')
kt_lte = raw.str.extract(r'(?:до|<)\s*(\d+(?:[.,]\d+)?)')
kt_mid = (pd.to_numeric(kt_range[0], errors='coerce') + pd.to_numeric(kt_range[1], errors='coerce')) / 2
kt_less = pd.to_numeric(kt_lte[0], errors='coerce')
kt_percent = kt_point.where(kt_point >= 1, kt_point * 100).fillna(kt_mid).fillna(kt_less)
kt_std = (kt_percent - kt_percent.mean()) / kt_percent.std()
print('КТ (відношень): після узгодження одиниць n =', int(kt_percent.notna().sum()),
      '| min', round(kt_percent.min(), 2), '| max', round(kt_percent.max(), 2),
      '| std', round(kt_percent.std(), 2))
print('КТ z-score: mean', round(kt_std.mean(), 3), '| std', round(kt_std.std(), 3))

sup_clean = df['Супутній'].dropna().astype(str).str.strip()
sup_dummies = pd.get_dummies(sup_clean, prefix='Супутній', dtype=int)
print('Супутній (номінальна): one-hot колонок', sup_dummies.shape[1],
      '| рядків із заповненим діагнозом', int((sup_dummies.sum(axis=1) > 0).sum()))
print('найрідша категорія трапляється', int(sup_clean.value_counts().min()), 'раз(и)')

ЛН (порядкова) -> коди із заданим порядком: {'І': 1, 'І-ІІ': 2, 'ІІ': 3, 'ІІ-ІІІ': 4, 'ІІІ': 5, 'ІІІ-IV': 6}
розподіл кодів: {1: 12, 2: 9, 3: 21, 4: 59, 5: 17, 6: 3}
КТ (відношень): після узгодження одиниць n = 43 | min 5.0 | max 80.0 | std 21.97
КТ z-score: mean -0.0 | std 1.0
Супутній (номінальна): one-hot колонок 90 | рядків із заповненим діагнозом 115
найрідша категорія трапляється 1 раз(и)


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (1, 'середній результат', 'середнє бінарної номінальної ознаки, нічого не вимірює', 'частки померлих і живих, мода'),
    (2, 'стать', 'змінна має кілька написань однієї статі, неуніфікована', 'уніфікувати написання статі'),
    (3, 'середній ступінь', 'середнє порядкової шкали та довільно закодованих рангів', 'медіана й мода із заданим порядком'),
    (4, 'кт', 'to_numeric coerce мовчки відкидає рядкові диапазони', 'уніфікувати формати й одиниці КТ'),
    (5, 'сатурація', 'середнє пораховане лише по підмножині непорожніх', 'частка пацієнтів нижче клінічного порогу'),
    (6, 'дата', 'коефіцієнт варіації дат не має змісту для timestamp', 'перехід до тривалостей або розмах дат'),
    (7, 'стать корр з помер', 'Пірсон для номінальних змінних некоректний', 'коефіцієнт Крамера V з хі-квадрат'),
    (8, 'нормалізація числових ознак', 'нормалізуються сталі, ідентифікатори, дати й цільова', 'відсіяти неналежні стовпці перед нормалізацією'),
    (9, 'dropna', 'dropna видаляє більшість рядків і руйнує вибірку', 'заповнювати пропуски або видаляти лише дати'),
    (10, 'рейтинг важливості (дата розтину)', 'дата розтину відома лише для померлих, витік цілі', 'видалити дати розтину з ознак моделі'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,середній результат,"середнє бінарної номінальної ознаки, нічого не вимірює","частки померлих і живих, мода"
1,2,стать,"змінна має кілька написань однієї статі, неуніфікована",уніфікувати написання статі
2,3,середній ступінь,середнє порядкової шкали та довільно закодованих рангів,медіана й мода із заданим порядком
3,4,кт,to_numeric coerce мовчки відкидає рядкові диапазони,уніфікувати формати й одиниці КТ
4,5,сатурація,середнє пораховане лише по підмножині непорожніх,частка пацієнтів нижче клінічного порогу
5,6,дата,коефіцієнт варіації дат не має змісту для timestamp,перехід до тривалостей або розмах дат
6,7,стать корр з помер,Пірсон для номінальних змінних некоректний,коефіцієнт Крамера V з хі-квадрат
7,8,нормалізація числових ознак,"нормалізуються сталі, ідентифікатори, дати й цільова",відсіяти неналежні стовпці перед нормалізацією
8,9,dropna,dropna видаляє більшість рядків і руйнує вибірку,заповнювати пропуски або видаляти лише дати
9,10,рейтинг важливості (дата розтину),"дата розтину відома лише для померлих, витік цілі",видалити дати розтину з ознак моделі


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# ваш код: три правильні обчислення замість хибних
# 1
частка = (df['помер/жив'] == 1).mean()
df['помер/жив'].value_counts(normalize=True)
# 2
стать = (df['Стать'].astype(str).str.strip().str.lower()
           .str.replace('жін', 'ж').str.replace('чол', 'ч')
           .str[0].map({'ч': 'чоловіча', 'ж': 'жіноча'}))
стать.value_counts()
# 7
табл = pd.crosstab(df['Стать'], df['помер/жив'])
from scipy.stats import chi2_contingency
chi2, p, dof, _ = chi2_contingency(табл)
V = (chi2 / (табл.values.sum() * (min(табл.shape) - 1))) ** 0.5

---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'КТ записано неузгоджено ("20-30%", "до 17%", 0.3), тому числові статистики обчислюються лише на 43 із 121 рядка і є зміщеними',
    'Дата розтину відома тільки для померлих (61 рядок), тому вона протікає цільову змінну й не може бути ознакою моделі',
    'Супутній - номінальний вільний текст: 91 унікальний діагноз на 115 записів, тому one-hot дає 90 розріджених ознак, непридатних без групування',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. КТ записано неузгоджено ("20-30%", "до 17%", 0.3), тому числові статистики обчислюються лише на 43 із 121 рядка і є зміщеними
2. Дата розтину відома тільки для померлих (61 рядок), тому вона протікає цільову змінну й не може бути ознакою моделі
3. Супутній - номінальний вільний текст: 91 унікальний діагноз на 115 записів, тому one-hot дає 90 розріджених ознак, непридатних без групування


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = """
1. розбити на 2 стовпця через дріб (сист/діаст),
2. залишити лише сист або діаст,
3. визначити середнє між двома і записати в стовпець.
Я б вибрав перший варіант, бо він зберігає найбільше даних про артеріальний тиск
"""
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


1. розбити на 2 стовпця через дріб (сист/діаст),
2. залишити лише сист або діаст,
3. визначити середнє між двома і записати в стовпець.
Я б вибрав перший варіант, бо він зберігає найбільше даних про артеріальний тиск



---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'код, перевірено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'роботу коду, написаного ШІ, синтаксис, читабельність коду, \
                       коректність та валідність відповідей, написаних ШІ'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 2 готова до захисту.
Михайло Білоусов, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.